# Step 19.5 — Baseline training run：先读懂协议，再决定是否运行

在仓库根目录同步母版：

```bash
git pull
```

再创建独立工作副本，不覆盖已有笔记：

```bash
cp -n notebooks/86_step19_5_baseline_training_run.ipynb notebooks/work/86_step19_5_baseline_training_run_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

以上是未来命令参考，本次未执行。环境仍是项目Python 3.12 / `.venv`、固定 **MLX v0.32.2**。本节名称中的run不是本轮训练授权：全部代码单元未执行、输出为空，没有读取语料、训练、验证或测量结果。

2026-11-17 教学重做：**历史核心修正2/2＋按本次新指令进行教学修订1次，累计3次**。不重置历史额度。只整理19.5的讲解、重复依赖和未来入口，保留现有训练协议及记录字段，不改其他小节。


## 阅读路线：本节新增的是一份可解释的运行协议

[19.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/82_step19_1_hyperparameter_baseline.ipynb)给出候选，[19.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/83_step19_2_train_validation_loop.ipynb)区分训练／验证及参数版本，[19.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/84_step19_3_training_stability.ipynb)在更新前检查梯度，[19.4](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/85_step19_4_apple_silicon_resource_control.ipynb)区分内存统计范围。本节把这些组织为一份有界的baseline protocol，并说明将来每个数字从哪里来。

| 区域 | 用途 | 本次是否运行？ |
|---|---|---|
| 正文1–7 | 候选、时间线、计时窗口、失败边界、记录字段 | 不运行；表格是流程／手算，不是日志 |
| 附录A/B | 已学数据、模型、loss和稳定性依赖 | 只定义；完整模型只保留一套，可按引用回顾 |
| 附录C | 原协议与记录函数的完整实现 | 只定义，不在单元末尾调用 |
| 附录D | 数据准备→完整setup→运行并写文件 | 全部为注释；未来主动使用才有副作用 |

保留原函数接口和同一Notebook上下文，不把协议改造成另一套训练框架。附录D完整提供所需数据与状态变量，不能用旧kernel里碰巧存在的同名变量替代。理解正文不需要先运行附录。


## 1. 候选不是调优结果，warm-up也算更新

原候选保持C=64、N=2、H=4、T=64、B=4、学习率3e-4，模型位置容量max_seq_len由block_size映射。真实V由真实字符表决定。计划仍是**1次warm-up更新＋20次计时更新；每5次计时更新验证一次，每次4个validation batches**。

与19.3的可选裁剪不同，这份既有基线明确设clip_max_norm=1.0，并在每步使用全局范数裁剪；它是原协议选择，不是经过本轮实验得到的最佳阈值。Adam仍显式bias_correction=True、betas=[0.9,0.999]、eps=1e-8。

下面只是Python配置。model seed控制模型初始化；持久train RNG控制训练窗口；每次验证另按validation seed建立局部RNG。运行到记录写完之前，保持同一份配置、数据和匹配setup，不中途改字典再把新配置标在旧metrics上。


In [ ]:
baseline_candidate = {
    "model_width": 64, "layers": 2, "heads": 4,
    "block_size": 64, "batch_size": 4, "learning_rate": 3e-4,
}
loop_plan = {"num_updates": 20, "eval_every": 5, "val_batches": 4}
seeds = {"model": 2028, "train": 2026, "validation": 2027}
clip_max_norm = 1.0


## 2. 先看版本时间线：本协议没有初始验证

θ[k]表示完成k次Adam更新后的参数。原19.2演示循环有初始验证；**本节run_baseline_protocol没有在θ[0]或warm-up后立即验证**。不能把前节的行为直接填进本节日志，也不为补一行表而改协议。

| 阶段 | 本次train loss使用的参数 | 完成后的参数／总更新数 | 是否验证？ |
|---|---|---|---|
| 完整新setup | 尚无train loss | θ[0]，Adam step=0 | 否；只检查fresh状态 |
| 1次warm-up | θ[0] | θ[1]，总数1 | 否；loss不放入20项训练曲线 |
| 计时更新1 | θ[1] | θ[2]，总数2 | 否 |
| 计时更新5 | θ[5] | θ[6]，总数6 | 是，固定θ[6]抽4批 |
| 计时更新10 | θ[10] | θ[11]，总数11 | 是 |
| 计时更新15 | θ[15] | θ[16]，总数16 | 是 |
| 计时更新20 | θ[20] | θ[21]，总数21 | 是，周期与末次重合只做一次 |

所以正常完成时，validation_points的完成数为 **6/11/16/21**。条件使用计时更新索引 `update_index % eval_every == 0`，记录标签再加warm-up的1；最后一次用or兜底，并不要求它恰好落在周期上。若未来改变计划，要按同一规则解释记录，不能继续硬写这四个标签。

`total_optimizer_updates`按1+记录到的计时步数计算，不是每步重新读取Adam step的独立核验。它依赖stable_train_step恰好更新一次的契约；不能替换成只forward或更新两次的函数而保留标签。原入口只在开始时核对Adam step=0。


### 对一条记录标注参数、数据与数组位置

第5次计时更新对应列表下标4：`training_loss_pre_update[4]`是θ[5]在该训练batch上的loss；随后第一条validation_points使用θ[6]及验证split。它们既不是同一参数，也不是同一批数据，不应相减当作这次更新带来的loss变化。

训练loss由raw logits (B,T,V)与已shift的Y (B,T)配对展平、cross-entropy后mean，返回标量()。验证直接调用同一loss_fn、不调用求导或optimizer，4批期间模型参数固定；按监督位置数加权。当前每批256个位置，4批合计1024个计入平均的位置，不代表1024个互不重叠的唯一token。

先90/10顺序划分，再在各自split内取T+1项；两边都至少65个token。训练RNG持续消耗，验证每次按固定seed重建局部RNG；在输入及B/T/批数相同时复用同一组验证窗口，不消耗训练RNG，也不是遍历整个验证集。


## 3. 计时的是哪一种训练步？

沿用[19.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/84_step19_3_training_stability.ipynb)的主机侧稳定性路径：计算旧参数loss和gradient tree→全局范数→求值并检查loss/G有限→统一裁剪→检查preclip norm→一次Adam.update→求值当前training_state。它是**未编译（uncompiled）**单步，不能把图内放不下的Python检查描述成已经被compile优化。

stable_train_step返回四项：loss_before_update、grad_norm、preclip_norm、clip_scale。前两种norm字段都是同一原梯度树的裁剪前范数；第二个来自clip helper的再次计算，不是裁剪后范数。clip_scale是统一缩放系数，不是学习率。

所有非有限loss/梯度拒绝发生在Adam前。代码没有loss大小阈值、温度／Swap自动停止、更新后参数或Adam状态的有限性检查，也没有对validation loss调用require_finite。有限值检查通过不保证收敛或新状态仍有限；计时完成更不表示训练质量合格。


## 4. 两个时钟窗口不能混用

`time.perf_counter()`给单调高分辨率计时器读数；差值是墙钟经过时间，单位秒。MLX有惰性求值，计时必须覆盖所要测量工作真正完成的边界，不能只量建立依赖的时间。

原循环按下面顺序；这是源码摘录，不是另一个可运行入口：

```python
X, Y, _ = get_batch(...)
mx.eval(X, Y)                         # 在单步计时之前准备好输入
t0 = time.perf_counter()
loss, grad_norm, preclip_norm, scale = stable_train_step(X, Y)
elapsed = time.perf_counter() - t0    # 单步内部已求值新状态
```

| 口径 | 包含 | 排除 |
|---|---|---|
| 每步elapsed | forward/loss、autodiff、主机有限值检查及其.item()、clipping、Adam update、最终state求值、函数内Python开销 | 该步采样、X/Y预先求值、外部列表追加与记录.item()、validation、warm-up |
| measured_loop_wall_seconds_including_validation_excluding_warmup | 20步主循环：采样、X/Y求值、每步训练、外部记录、周期/末次validation | warm-up、起点计数/peak reset、循环结束后的内存读数、build record与JSON写入 |

这两个口径都不是整份Notebook从读取数据开始的总耗时。setup的模型初始化／初始求值也在计时外。单步内部已有mx.eval(training_state)，所以停止计时前要求完成此次新状态；它不是更新后数值健康检查。

`updates_per_second_excluding_sampling_validation_and_warmup = N / sum(step_seconds)`，正常N=20；不是N除以整段loop耗时，也不是各步1/elapsed的算术平均，更不是tokens/s。纯纸算：若两步分别耗2秒与1秒，总速率是2/3 updates/s，而非(1/2+1)/2。本节没有测得这两个时间。


### 内存窗口与计时窗口也不同

一次warm-up已更新并求值后，先读active_start/cache_start，再reset_peak_memory，然后进入20步循环；结束后才读active_end/cache_end及raw_peak_after_reset。这沿用[18.5](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/81_step18_5_memory_instrumentation.ipynb)和[19.4](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/85_step19_4_apple_silicon_resource_control.ipynb)的统计口径：

| 字段 | 含义 |
|---|---|
| active_start_bytes / cache_start_bytes | warm-up后、计时循环前的MLX分配器起点 |
| active_end_bytes / cache_end_bytes | 整个循环及验证结束后的读数 |
| raw_peak_after_reset_bytes | reset之后记录到的active峰值 |
| window_peak_active_bytes | max(active_start,raw_peak)，把起点已存活内存算入窗口高点 |
| active_delta_bytes | active_end−active_start，可正可负；不是峰值增量 |

peak窗口包括20步采样／输入、训练和validation中的分配，不是仅stable_train_step的峰值。warm-up本身在reset之前，但它留下的模型、Adam、仍被引用的数组以及可复用buffer可能延续到后面的窗口；reset不清空它们。原函数没有每步memory snapshot列表，也没有清cache或修改资源限制。

这些数以bytes计，只描述MLX分配器，不等于整机RAM、温度、Swap或Memory Pressure。不能把active+cache+peak相加当总内存，不能从active_delta小推断没有内存压力。


## 5. 单次使用、失败和停止边界

协议使用 `baseline_protocol_used` 与Adam step一起防止把旧状态重跑后再次标成21步。它们与整组model、optimizer、求导函数、training_state、train_rng一同在完整setup中准备。

| 时点 | 真实检查或动作 | 失败之后怎样？ |
|---|---|---|
| 入口首先 | 若used已为True，报错 | 在新采样／新更新前拒绝 |
| 仍未用过时 | 求值并读取Adam step，必须为0 | 非0就报错，尚不设置used=True |
| 首次采样前 | 立即used=True | 后面的失败不会自动清除标记 |
| warm-up或任一计时步 | 执行更新前finite检查 | 异常中止调用；之前已完成的更新不回滚 |
| 正常结束 | 返回metrics | 同一组状态仍不能再次调用协议 |

中途报错／手动中断时，可能已有部分更新和RNG消耗；没有返回“成功完成”的metrics，也不会自动保存局部列表或checkpoint。Python顺序入口中的build_baseline_record与写入只有run正常返回才会继续。不能补造完整20步记录、只把used改回False或只重设seed后宣称是全新baseline。

若未来确需重试一次全新基线，重新完整创建model/Adam/求导函数/state/train_rng和标记；这是新运行，不是从中断处恢复。仅重新定义run_baseline_protocol不会重置标记。原代码没有自动恢复、异常后清理、墙钟超时或系统压力监控；人工停止也是停止继续工作，不承诺撤销已发生的更新。

当前计划是有界短候选，不证明持续负载安全。若未来出现真实异常或系统压力，应停止追加工作并记录情况；本节不自动增加步数来追求更好结果。


## 6. Record schema：字段是什么，不是什么

build_baseline_record(metrics)只是组装schema_version=1的Python字典，不创建权重checkpoint，也不是完整恢复训练的容器：

| 顶层字段 | 记录内容与边界 |
|---|---|
| schema_version / created_utc | 格式版本1；组装record时的UTC时间，不是训练起始时刻 |
| config | 候选、20/5/4计划、clip_max_norm的合并；不自动包含Adam全部超参数 |
| seeds | model/train/validation种子；不是当前RNG完整状态 |
| data | 缓存绝对路径、文件SHA-256、顺序split规则/索引、两段长度、vocab_size；不是词表映射或语料副本 |
| environment | Python/平台/machine、mx.__version__、default_device；不等于完整机器配置与当时负载 |
| timing_definition | 上述计时包含／排除范围，以文字与标志明确记下 |
| metrics | 原协议正常返回的实际测量与曲线；不能填纸算例冒充结果 |
| system_observations | 温度、Swap、memory_pressure；代码不采集，默认None |

数据SHA-256用于标识读取的缓存文件内容，不替代训练时保持文件不变。配置、数据、软件环境与计时口径让未来比较有据可查，但不能保证跨设备／系统状态逐bit重现；文件也没有模型权重、Adam m/v或完整sampler状态。


### Metrics列表怎样对应同一次更新？

| 字段／组 | 正常完成时的结构 |
|---|---|
| warmup_updates / timed_updates / total_optimizer_updates | 1、20、21，前提是原单步每次恰好一次更新 |
| training_loss_pre_update | 20项，按计时更新1…20排列；warm-up loss不在里面 |
| gradient_norm_preclip / gradient_norm_used_for_clipping / clip_scale | 各20项，与同下标train loss和step time对齐；两种norm都在裁剪前 |
| step_wall_seconds_uncompiled_step_including_host_checks_clipping_update_and_state_eval | 20项秒数，口径见前文 |
| validation_points | 4个字典，键为completed_updates和validation_loss_current_parameters；总更新6/11/16/21 |
| measured_loop_wall_seconds_including_validation_excluding_warmup | 一枚整段循环秒数 |
| updates_per_second_excluding_sampling_validation_and_warmup | N/sum(step_seconds)；若无步数或时间和非正则None |
| *_bytes | 起点／终点／窗口高点的内存整数，不是每步序列 |

Python None经json.dump写为JSON null，含义是未测／不可计算，**不是0**。system_observations的三项默认保持null；没有实际运行时连metrics都不应伪造出来。没有被记录的warm-up loss或初始validation不能自行补一个0。

原build函数不验证metrics，json.dump也不证明数值有效；其默认设置可输出NaN/Infinity，validation又没有finite guard，因此不能把成功写出文件等同于严格有效的数值记录。本次保持原协议与序列化接口，不凭空增加运行结果或称它已通过数据schema验证。


## 7. 安全保存和阅读检查点

未来正常完成后才调用build_baseline_record，再写到 `results/baseline/baseline_<UTC时间戳>.json`。`open("x")`是独占创建：同名文件已存在时抛FileExistsError，不覆盖旧记录。时间戳以秒为单位仍可能碰撞，真正防覆盖的是x模式。

这不是原子事务或自动备份；若写入中断，可能留下不完整的新文件，不能把它当作成功记录。本协议没有自动重试、覆盖或删旧文件的路径。Notebook工作副本也继续用时间戳加cp -n保护笔记。

自查（不附答案）：

1. 用自己的话解释第10次计时更新的loss、当前参数和验证标签分别如何对应。
2. 若只比较updates_per_second而忽略validation和采样开销，遗漏了哪些工作？
3. 报错后为什么不能只把used改回False，再将旧模型记为新的21步baseline？

以下是完整旧依赖及实际协议定义。没有真实运行时，前面的表都是对代码契约的解释，不是已完成的测试或训练日志。


## 附录A · 已学数据与稳定性依赖

配置映射沿用[19.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/82_step19_1_hyperparameter_baseline.ipynb)；数据／split／sampler见[16.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/67_step16_1_acquire_inspect_tiny_shakespeare.ipynb)、[16.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/68_step16_2_train_validation_split.ipynb)、[16.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/69_step16_3_batch_sampler.ipynb)。这些函数只定义，未来调用才读文件或采样；默认获取关闭，缓存缺失／split太短就停止，不换toy数据。


In [ ]:
import hashlib
import json
import math
import platform
import time
from datetime import datetime, timezone
from pathlib import Path
import random
from urllib.request import urlopen

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from mlx.utils import tree_flatten, tree_map


In [ ]:
def make_model_kwargs(config, vocab_size):
    return {
        "vocab_size": vocab_size, "dims": config["model_width"],
        "num_heads": config["heads"], "num_blocks": config["layers"],
        "max_seq_len": config["block_size"],
    }

def make_sampler_kwargs(config):
    return {"batch_size": config["batch_size"], "block_size": config["block_size"]}

def make_optimizer_kwargs(config):
    return {
        "learning_rate": config["learning_rate"], "betas": [0.9, 0.999],
        "eps": 1e-8, "bias_correction": True,
    }


In [ ]:
SOURCE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Repository root not found; set PROJECT_ROOT explicitly.')

def acquire_tiny_shakespeare(path, source_url=SOURCE_URL):
    path = Path(path)
    if path.is_file():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(source_url, timeout=30) as response:
        raw = response.read()
    tmp = path.with_suffix('.tmp')
    tmp.write_bytes(raw)
    tmp.replace(path)
    return path


loader建立同一字符表与整数token stream，split按90/10顺序切开；sampler只接已选中的一段，返回X/Y各(B,T)。Y已shift，不在loss里再移一次。函数保持原接口，仅把紧挤的一行写法展开方便阅读。


In [ ]:
def load_cached_character_corpus(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Cache missing: {path.resolve()}; correct path or explicitly acquire it.')
    text = path.read_text(encoding='utf-8')
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    return (text, chars, stoi, itos, mx.array([stoi[ch] for ch in text], dtype=mx.int32))

def split_train_validation(stream, fraction=0.9):
    if len(stream.shape) != 1 or stream.shape[0] < 2 or (not 0 < fraction < 1):
        raise ValueError('Expected 1-D stream and 0<fraction<1')
    s = int(stream.shape[0] * fraction)
    if s <= 0 or s >= stream.shape[0]:
        raise ValueError('Empty split')
    return (stream[:s], stream[s:], s)


In [ ]:
def get_batch(data, batch_size, block_size, rng):
    if len(data.shape) != 1 or type(batch_size) is not int or batch_size < 1 or (type(block_size) is not int) or (block_size < 1):
        raise ValueError('Invalid sampler inputs')
    n = data.shape[0]
    if n < block_size + 1:
        raise ValueError('Split needs block_size+1 tokens')
    starts = [rng.randrange(0, n - block_size) for _ in range(batch_size)]
    xs = []
    ys = []
    for s in starts:
        chunk = data[s:s + block_size + 1]
        xs.append(chunk[:-1])
        ys.append(chunk[1:])
    return (mx.stack(xs, axis=0), mx.stack(ys, axis=0), starts)


### 原bootstrap收进主动入口

prepare_cached_data(project_root,config,acquire_missing_data=False)保留原缓存路径、真实词表、90/10划分、T+1检查、文件SHA-256与token求值。只是定义不会读取；调用时才发生这些动作，并把所需字段放入字典。附录D显式恢复原协议使用的变量，不依赖旧kernel。


In [ ]:
def prepare_cached_data(project_root, config, *, acquire_missing_data=False):
    PROJECT_ROOT = find_project_root(project_root)
    CACHE_PATH = PROJECT_ROOT / 'data' / 'tinyshakespeare' / 'input.txt'
    if acquire_missing_data:
        acquire_tiny_shakespeare(CACHE_PATH)
    text, characters, stoi, itos, token_stream = load_cached_character_corpus(CACHE_PATH)
    vocab_size = len(characters)
    train_tokens, val_tokens, split_index = split_train_validation(token_stream, 0.9)
    required_tokens = config['block_size'] + 1
    for split_name, split_data in (('train', train_tokens), ('validation', val_tokens)):
        if split_data.shape[0] < required_tokens:
            raise ValueError(f'{split_name} split needs at least {required_tokens} tokens.')
    data_sha256 = hashlib.sha256(CACHE_PATH.read_bytes()).hexdigest()
    mx.eval(train_tokens, val_tokens)
    return {
        "PROJECT_ROOT": PROJECT_ROOT,
        "CACHE_PATH": CACHE_PATH,
        "text": text,
        "characters": characters,
        "stoi": stoi,
        "itos": itos,
        "token_stream": token_stream,
        "vocab_size": vocab_size,
        "train_tokens": train_tokens,
        "val_tokens": val_tokens,
        "split_index": split_index,
        "data_sha256": data_sha256,
    }


### 稳定性helper是复用，不是新算法

[19.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/84_step19_3_training_stability.ipynb)已说明gradient tree→平方和→全局L2范数与统一缩放。保留本节已有global_grad_norm、require_finite、clip_grad_tree接口；输入为原实数浮点模型的数组梯度树。clip_max_norm=1.0是有限正候选；helper不是通用参数校验器，也不应给它NaN/Inf阈值。

require_finite在主机侧求值并读取布尔判断，失败抛异常；裁剪返回新树、裁剪前norm与scale。这些定义不会计算任何真实梯度。


In [ ]:
def global_grad_norm(grad_tree):
    leaves = tree_flatten(grad_tree)
    squared_sums = [mx.sum(mx.square(leaf)) for _, leaf in leaves]
    if not squared_sums:
        raise ValueError("Gradient tree has no array leaves")
    return mx.sqrt(sum(squared_sums))

def require_finite(value, label):
    mx.eval(value)
    if not bool(mx.all(mx.isfinite(value)).item()):
        raise FloatingPointError(f"{label} is NaN or Inf")


In [ ]:
def clip_grad_tree(grad_tree, max_norm, eps_clip=1e-12):
    if max_norm <= 0:
        raise ValueError("max_norm must be positive")
    norm = global_grad_norm(grad_tree)
    mx.eval(norm)
    scale = mx.minimum(
        mx.array(1.0, dtype=mx.float32),
        mx.array(max_norm, dtype=mx.float32) / (norm + eps_clip),
    )
    return tree_map(lambda grad: grad * scale, grad_tree), norm, scale


## 附录B · 完整旧模型只保留一套

原403行块的三个底层类各出现三次，最后实际生效版本与[15.4完整模型](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/65_step15_4_implement_minigpt.ipynb)、已验收19.2/19.3相同。现各留一份，不改变模型层数、bias、Pre-Norm RMSNorm、因果mask、C→4C→C MLP、独立Block或untied head。

这是可选前置汇总，不是19.5新架构。MHA输入／输出(B,T,C)，score为(B,H,T,T)，形状推导见[13.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/54_step13_3_split_channels_into_heads.ipynb)；以下只定义类。


In [ ]:
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)


### MLP与Block

沿用[14.4](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/60_step14_4_feed_forward_mlp.ipynb)的带bias两层MLP与GELU。两条Pre-Norm残差路径保持(B,T,C)，独立Block实例由MiniGPT中的num_blocks决定，不由文本重复定义次数决定。


In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


### MiniGPT汇总

整数IDs (B,T)经token/learned-position embedding、N个Block和final RMSNorm，输出raw logits (B,T,V)。V来自真实字符表，T受位置容量限制；保留原检查与参数结构，不自动构造实例。


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


### 复用loss与验证helper

[17.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c2167852cdb48ce3b0d145657e963c6dade3fccf/notebooks/73_step17_2_flatten_bt_supervision.ipynb)的flatten与逐位置交叉熵契约不变。验证期间固定当前参数、独立RNG、按位置加权；model.eval只切模式，finally恢复原模式，不代表关闭梯度。验证没有optimizer更新路径。


In [ ]:
def loss_fn(model, X, Y):
    # X, Y: (B, T); Y is already shifted by get_batch.
    logits = model(X)  # (B, T, V), raw unnormalized logits
    B, T, V = logits.shape
    flat_logits = mx.reshape(logits, (B * T, V))  # (B*T, V)
    flat_targets = mx.reshape(Y, (B * T,))        # (B*T,)
    token_losses = nn.losses.cross_entropy(
        flat_logits, flat_targets, axis=-1, reduction="none"
    )  # (B*T,)
    return mx.mean(token_losses)


In [ ]:
def estimate_validation_loss(model, val_tokens, batch_size, block_size, num_batches, seed):
    if type(num_batches) is not int or num_batches < 1:
        raise ValueError('num_batches must be positive int')
    rng = random.Random(seed)
    was_training = model.training
    model.eval()
    try:
        total_loss = 0.0
        total_positions = 0
        for _ in range(num_batches):
            Xv, Yv, _ = get_batch(val_tokens, batch_size, block_size, rng)
            loss = loss_fn(model, Xv, Yv)
            mx.eval(loss)
            n = Yv.shape[0] * Yv.shape[1]
            total_loss += loss.item() * n
            total_positions += n
        return total_loss / total_positions
    finally:
        model.train(was_training)


## 附录C · 原协议完整实现，定义不运行

以下保留stable_train_step、run_baseline_protocol()、build_baseline_record(metrics)的实际接口与算法。它们使用本Notebook中的匹配上下文；附录D一次提供完整data/setup，不应单独替换model或optimizer后沿用旧求导函数／state。

stable_train_step保留四返回值与更新前检查顺序；不要将四项tuple直接当scalar调.item()，也不要把它整体放进mx.compile。此时只定义函数，未创建模型、求导函数或Adam。


In [ ]:
def stable_train_step(X, Y):
    # Host-side evaluation/checks occur before optimizer.update; this function is not compiled.
    loss_before_update, grads = loss_and_grad_fn(model, X, Y)
    grad_norm = global_grad_norm(grads)
    mx.eval(loss_before_update, grad_norm)
    require_finite(loss_before_update, "training loss")
    require_finite(grad_norm, "gradient norm")
    clipped_grads, preclip_norm, clip_scale = clip_grad_tree(grads, clip_max_norm)
    mx.eval(preclip_norm, clip_scale)
    require_finite(preclip_norm, "pre-clipping gradient norm")
    optimizer.update(model, clipped_grads)
    mx.eval(training_state)
    return loss_before_update, grad_norm, preclip_norm, clip_scale


### 对照正文时间线读run_baseline_protocol

前段是单次使用／step=0检查和warm-up；中段是20步计时与周期验证；末段只返回原字段。baseline_protocol_used只在附录D的完整新setup里设False，定义这个函数不会重置它。输入采样、主机检查、求值和日志位置与原协议一致。


In [ ]:
def run_baseline_protocol():
    global baseline_protocol_used
    # Reject re-entry before sampling or any update. Re-run the complete setup cell for a fresh run.
    if baseline_protocol_used:
        raise RuntimeError("Baseline protocol is single-use. Re-run the complete model/Adam/RNG setup cell first.")
    mx.eval(optimizer.state["step"])
    initial_step = int(optimizer.state["step"].item())
    if initial_step != 0:
        raise RuntimeError(f"Expected a fresh Adam state with step 0, got {initial_step}. Re-run the complete setup cell.")
    # Mark before the first sample; failures do not automatically clear this guard.
    baseline_protocol_used = True
    # One real warm-up update, explicitly excluded from the 20 timed updates.
    X0, Y0, _ = get_batch(train_tokens, baseline_candidate["batch_size"], baseline_candidate["block_size"], train_rng)
    mx.eval(X0, Y0)
    warm_loss, _, _, _ = stable_train_step(X0, Y0)
    mx.eval(warm_loss, training_state)

    active_start = mx.get_active_memory()
    cache_start = mx.get_cache_memory()
    mx.reset_peak_memory()
    step_seconds, train_losses, grad_norms, preclip_norms, clip_scales = [], [], [], [], []
    validation_points = []
    loop_start = time.perf_counter()

    for update_index in range(1, loop_plan["num_updates"] + 1):
        X, Y, _ = get_batch(train_tokens, baseline_candidate["batch_size"], baseline_candidate["block_size"], train_rng)
        mx.eval(X, Y)  # Exclude Python sampling and input materialization from the per-step timer.
        t0 = time.perf_counter()
        loss_before_update, grad_norm, preclip_norm, clip_scale = stable_train_step(X, Y)
        # stable_train_step completes loss, gradients, clipping and updated state before returning.
        elapsed = time.perf_counter() - t0
        step_seconds.append(elapsed)
        train_losses.append(loss_before_update.item())
        grad_norms.append(grad_norm.item())
        preclip_norms.append(preclip_norm.item())
        clip_scales.append(clip_scale.item())

        completed_updates = update_index + 1  # one warm-up update plus measured updates
        if update_index % loop_plan["eval_every"] == 0 or update_index == loop_plan["num_updates"]:
            val_loss = estimate_validation_loss(
                model, val_tokens, baseline_candidate["batch_size"],
                baseline_candidate["block_size"], loop_plan["val_batches"],
                seeds["validation"],
            )
            validation_points.append({
                "completed_updates": completed_updates,
                "validation_loss_current_parameters": val_loss,
            })

    measured_loop_seconds = time.perf_counter() - loop_start
    active_end = mx.get_active_memory()
    cache_end = mx.get_cache_memory()
    raw_peak_after_reset = mx.get_peak_memory()
    return {
        "warmup_updates": 1,
        "timed_updates": len(step_seconds),
        "total_optimizer_updates": 1 + len(step_seconds),
        "step_wall_seconds_uncompiled_step_including_host_checks_clipping_update_and_state_eval": step_seconds,
        "training_loss_pre_update": train_losses,
        "gradient_norm_preclip": grad_norms,
        "gradient_norm_used_for_clipping": preclip_norms,
        "clip_scale": clip_scales,
        "validation_points": validation_points,
        "measured_loop_wall_seconds_including_validation_excluding_warmup": measured_loop_seconds,
        "updates_per_second_excluding_sampling_validation_and_warmup": (
            len(step_seconds) / sum(step_seconds) if step_seconds and sum(step_seconds) > 0 else None
        ),
        "active_start_bytes": active_start,
        "cache_start_bytes": cache_start,
        "active_end_bytes": active_end,
        "cache_end_bytes": cache_end,
        "raw_peak_after_reset_bytes": raw_peak_after_reset,
        "window_peak_active_bytes": max(active_start, raw_peak_after_reset),
        "active_delta_bytes": active_end - active_start,
    }


### 原record组装函数

它从同一份config、seeds、缓存元数据与metrics构建字典，调用时才读取环境／当前UTC时间。没有执行时不会有真实record。请勿修改配置、改换语料或重建setup之后，再用旧metrics组装一份看似匹配的新记录。


In [ ]:
def build_baseline_record(metrics):
    return {
        "schema_version": 1,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "config": dict(baseline_candidate) | dict(loop_plan) | {"clip_max_norm": clip_max_norm},
        "seeds": dict(seeds),
        "data": {
            "cache_path": str(CACHE_PATH.resolve()),
            "sha256": data_sha256,
            "split_rule": "contiguous 90/10",
            "split_index": int(split_index),
            "train_tokens": int(train_tokens.shape[0]),
            "validation_tokens": int(val_tokens.shape[0]),
            "vocab_size": int(vocab_size),
        },
        "environment": {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "machine": platform.machine(),
            "mlx_version": mx.__version__,
            "device": str(mx.default_device()),
        },
        "timing_definition": {
            "step_timer": "perf_counter around uncompiled stable_train_step; includes host finite checks, clipping, Adam update, and explicit state evaluation; excludes sampling, pre-evaluated X/Y, validation, and warm-up",
            "warmup_updates_excluded": 1,
            "updates_per_second_excludes_validation_sampling_and_warmup": True,
            "measured_loop_wall_time_includes_validation": True,
        },
        "metrics": metrics,
        "system_observations": {
            "temperature_celsius": None,
            "swap_used_bytes": None,
            "memory_pressure": None,
            "note": "Only populate these from explicit system observations; never infer them from MLX counters."
        },
    }


## 附录D · 未来主动入口，全部保持注释

下面三段按顺序配套使用。所有依赖定义需先齐备；此处默认不运行、不读取语料、不初始化模型、不训练、不写文件。未来在独立工作副本主动取消注释时，才会发生所写动作。

第一段恢复原数据字段，第二段完整创建model／Adam／求导函数／state／train_rng并重置一次使用标记，第三段才实际运行并写结果。仅重跑第二段可在同一份已准备数据上开始一次全新状态；它不是断点续训，不能单独重置标记伪装新run。


In [ ]:
# 1. 未来明确选择的数据准备；本次不取消注释、不执行。
# PROJECT_ROOT = Path("/absolute/path/to/mlx-minigpt-from-scratch")
# ACQUIRE_MISSING_DATA = False
# data = prepare_cached_data(PROJECT_ROOT, baseline_candidate,
#                            acquire_missing_data=ACQUIRE_MISSING_DATA)
# PROJECT_ROOT, CACHE_PATH = data["PROJECT_ROOT"], data["CACHE_PATH"]
# text, characters = data["text"], data["characters"]
# stoi, itos, token_stream = data["stoi"], data["itos"], data["token_stream"]
# vocab_size = data["vocab_size"]
# train_tokens, val_tokens = data["train_tokens"], data["val_tokens"]
# split_index, data_sha256 = data["split_index"], data["data_sha256"]
# print("Vocabulary size:", vocab_size)
# print("Split shapes:", train_tokens.shape, val_tokens.shape)
# print("Data SHA-256:", data_sha256)


第二段保留原setup顺序：先配置与seed，再新model与求导函数、新Adam及初始化state，建立同一组原状态容器引用、持久train_rng、used=False，最后求值初始状态。这是唯一正常初始化单次使用标记的位置；不要只摘出最后几行重置旧对象。


In [ ]:
# 2. 未来完整新setup；会创建并求值模型/Adam，不是训练更新。
# model_kwargs = make_model_kwargs(baseline_candidate, vocab_size=vocab_size)
# sampler_kwargs = make_sampler_kwargs(baseline_candidate)
# adam_kwargs = make_optimizer_kwargs(baseline_candidate)
# mx.random.seed(seeds["model"])
# model = MiniGPT(**model_kwargs)
# model.train()
# loss_and_grad_fn = nn.value_and_grad(model, loss_fn)
# optimizer = optim.Adam(**adam_kwargs)
# optimizer.init(model.trainable_parameters())
# training_state = [model.state, optimizer.state]
# train_rng = random.Random(seeds["train"])
# baseline_protocol_used = False
# mx.eval(training_state)


第三段沿用原顺序和输出路径。run正常返回才会build/write；写入失败不覆盖已有文件，也不表示训练尚未发生。这个入口没有try/except自动重试。下面的保存消息只能来自未来实际成功写入，本次没有打印它或产生文件。


In [ ]:
# 3. 未来主动训练、验证、测量并写JSON；本次不取消注释、不执行。
# metrics = run_baseline_protocol()
# record = build_baseline_record(metrics)
# results_dir = PROJECT_ROOT / "results" / "baseline"
# results_dir.mkdir(parents=True, exist_ok=True)
# stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
# record_path = results_dir / f"baseline_{stamp}.json"
# with record_path.open("x", encoding="utf-8") as f:
#     json.dump(record, f, ensure_ascii=False, indent=2)
# print("Saved baseline record:", record_path)


## 版本依据与本次交付

- [MLX v0.32.2 memory.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/memory.cpp)：active/cache/peak以bytes计，reset重置计数而非清空存活数据。
- [v0.32.2 Optimizer/Adam](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/optimizers/optimizers.py)：update推进状态，保留课程显式Adam配置。
- [v0.32.2 utils.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/utils.py)：tree_flatten/tree_map接口。

本次保留原训练协议、四项单步返回值、验证标签、计时范围、record字段和x模式保存。去掉重复模型定义，数据bootstrap改为主动函数，原完整setup与运行/写入改为带说明的注释入口；没有引入新训练算法或伪日志。

全部代码单元execution_count=null、outputs=[]。仅做JSON/schema、唯一合法ID、Python语法、依赖及原算法AST静态核对；没有执行Notebook、配置/测试单元、训练、验证、实验、采样、测量、语料读取下载、checkpoint或用户工作副本。未运行不能写成运行通过，教材交付不等于学员掌握。

19.5保留历史核心修正2/2，本次新指令下教学修订另计1次，累计3次；其他历史计数不变。本轮只完成19.5，停下等待独立验收。
